# Graded Codio Activity: Estimating a Multinomial Logit Mode-Choice Model
In this graded activity you will estimate a __multinomial (conditional) logit__ model of how travelers choose among four modes — `air`, `train`, `bus`, and `car` — for intercity trips, using the classic `TravelMode` dataset of 210 travelers. You will implement the choice probabilities and the log-likelihood, estimate the coefficients by maximum likelihood, and interpret the result with an elasticity.

> __Learning Objectives.__
>
> At the end of this activity, students will be able to:
> * __Choice probabilities:__ Implement the multinomial logit choice probabilities as a softmax of the deterministic utilities.
> * __Maximum likelihood:__ Implement the conditional-logit log-likelihood and estimate the coefficients by maximizing it.
> * __Interpretation:__ Read the estimated coefficients and compute an own-cost elasticity of a choice probability.

> __Grading.__
> This activity is autograded. Complete each `# TODO` in the __Task__ cells and run the notebook top to bottom. Your `choice_probabilities`, `negloglikelihood`, the estimated coefficients `θ̂`, and `elasticity_car` are checked against the reference solution.

Let's get started.
___

## Setup, Data, and Prerequisites
The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages (including `CSV`, `DataFrames`, `Optim`, and `ForwardDiff`), and includes our source code. The first run may take a few minutes.

In [1]:
include(joinpath(@__DIR__, "Include.jl"));

  Activating project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/CHEME-140-eCornell-Repository/courses/CHEME-145/module-1`


### Data
The `TravelMode` dataset is in __long format__: one row per traveler per mode, four rows per traveler. `choice` marks the mode actually taken. The utility uses two alternative-varying attributes: `gcost` (generalized cost) and `wait` (terminal waiting time, which is 0 for `car`). `income` and `size` are traveler-specific and are not used in this specification.

In [2]:
df = CSV.read(joinpath(_PATH_TO_DATA, "TravelMode.csv"), DataFrame);
first(df, 8)

Row,rownames,individual,mode,choice,wait,vcost,travel,gcost,income,size
,Int64,Int64,String7,String3,Int64,Int64,Int64,Int64,Int64,Int64
1,1,1,air,no,69,59,100,70,35,1
2,2,1,train,no,34,31,372,71,35,1
3,3,1,bus,no,35,25,417,70,35,1
4,4,1,car,yes,0,10,180,30,35,1
5,5,2,air,no,64,58,68,68,30,2
6,6,2,train,no,44,31,354,84,30,2
7,7,2,bus,no,53,25,399,85,30,2
8,8,2,car,yes,0,11,255,50,30,2


We reshape the long table into per-traveler arrays: `gcost` and `wait` are `N × J` matrices (one row per traveler, one column per mode), and `chosen` holds the index of each traveler's chosen mode. The mode order is fixed as `air, train, bus, car`, with `car` as the reference alternative (index 4).

In [3]:
modes = ["air", "train", "bus", "car"];      # car is the reference alternative (index 4)
modeidx = Dict(modes[j] => j for j in 1:length(modes));
individuals = unique(df.individual);
N = length(individuals);   # number of travelers
J = length(modes);         # number of alternatives

gcost  = Matrix{Float64}(undef, N, J);   # generalized cost of each mode
wait   = Matrix{Float64}(undef, N, J);   # terminal wait time of each mode
chosen = Vector{Int}(undef, N);          # index of the chosen mode
for (i, ind) in enumerate(individuals)
    sub = df[df.individual .== ind, :]
    for r in eachrow(sub)
        j = modeidx[r.mode]
        gcost[i,j] = r.gcost
        wait[i,j]  = r.wait
        r.choice == "yes" && (chosen[i] = j)
    end
end
println("loaded $N travelers × $J modes");

loaded 210 travelers × 4 modes


### Model
Each traveler $i$ receives utility $U_{ij}=V_{ij}+\varepsilon_{ij}$ from mode $j$, with deterministic part
$$
V_{ij} = \text{ASC}_{j} + \beta_{\text{gcost}}\,\text{gcost}_{ij} + \beta_{\text{wait}}\,\text{wait}_{ij},
$$
where $\text{ASC}_{j}$ is an alternative-specific constant and $\text{ASC}_{\text{car}}\equiv 0$ (car is the reference). Under IID Gumbel errors the choice probabilities are the multinomial logit
$$
P_{ij} = \frac{\exp(V_{ij})}{\sum_{k=1}^{J}\exp(V_{ik})},
$$
and the coefficients $\theta=(\text{ASC}_{\text{air}},\text{ASC}_{\text{train}},\text{ASC}_{\text{bus}},\beta_{\text{gcost}},\beta_{\text{wait}})$ are estimated by maximizing the log-likelihood
$$
\ell(\theta) = \sum_{i=1}^{N}\ln P_{i,\,c_i},
$$
where $c_i$ is the mode traveler $i$ chose.
___

## Task 1: Choice Probabilities
Complete `choice_probabilities(V)` so it returns the logit probabilities $P_{j}=\exp(V_{j})/\sum_k\exp(V_{k})$ for a vector `V` of deterministic utilities. Subtract `maximum(V)` before exponentiating for numerical stability (this does not change the result).

In [4]:
function choice_probabilities(V::AbstractVector)
    z = V .- maximum(V)          # numerical stability
    e = exp.(z)
    return e ./ sum(e)
end

# self-check: expect ≈ [0.475, 0.175, 0.175, 0.175]
choice_probabilities([1.0, 0.0, 0.0, 0.0])

4-element Vector{Float64}:
 0.4753668864186717
 0.17487770452710943
 0.17487770452710943
 0.17487770452710943

## Task 2: The Log-Likelihood
Complete `negloglikelihood(θ)`, the __negative__ log-likelihood. For each traveler the utility vector `V` is built for you; call your `choice_probabilities` and add the log-probability of the chosen mode `chosen[i]` to `ll`. The function returns `-ll` so we can minimize it.

In [5]:
function negloglikelihood(θ)
    asc = (θ[1], θ[2], θ[3], zero(eltype(θ)))   # ASC_car = 0
    bgc, bw = θ[4], θ[5]
    ll = zero(eltype(θ))
    for i in 1:N
        V = [asc[j] + bgc*gcost[i,j] + bw*wait[i,j] for j in 1:J]
        P = choice_probabilities(V)
        ll += log(P[chosen[i]])
    end
    return -ll
end

# self-check: at θ = 0 every mode is equally likely, so the value is N*log(4)
negloglikelihood(zeros(5))

291.12181583517753

## Task 3: Estimate the Model
This cell is __provided__. It minimizes your `negloglikelihood` with a gradient-based optimizer (the gradient comes from automatic differentiation) and reports the estimates. Run it after completing Tasks 1 and 2.

In [6]:
g!(storage, θ) = ForwardDiff.gradient!(storage, negloglikelihood, θ);
result = optimize(negloglikelihood, g!, zeros(5), BFGS());
θ̂ = Optim.minimizer(result);

coefnames = ["ASC_air", "ASC_train", "ASC_bus", "b_gcost", "b_wait"];
for k in 1:5
    println(rpad(coefnames[k], 10), " = ", round(θ̂[k], digits = 5));
end
println("\nlog-likelihood = ", round(-Optim.minimum(result), digits = 3));
println("cost and time coefficients both negative? ", θ̂[4] < 0 && θ̂[5] < 0);

ASC_air    = 5.77636
ASC_train  = 3.923
ASC_bus    = 3.21073
b_gcost    = -0.01578
b_wait     = -0.09709

log-likelihood = -199.977
cost and time coefficients both negative? true


## Task 4: Own-Cost Elasticity
The __own-cost elasticity__ of a mode's choice probability is the percentage change in the probability of choosing that mode when its own generalized cost rises by one percent. For the logit model the individual elasticity of mode $j$ with respect to its own cost is $\beta_{\text{gcost}}\,\text{gcost}_{ij}\,(1-P_{ij})$, and we report the sample average for `car`:
$$
E_{\text{car}} = \frac{1}{N}\sum_{i=1}^{N}\beta_{\text{gcost}}\,\text{gcost}_{i,\text{car}}\,(1-P_{i,\text{car}}).
$$
The fitted probabilities `Phat` are computed for you. Complete `elasticity_car`.

In [7]:
# fitted choice probabilities at θ̂ (provided)
Phat = zeros(N, J);
for i in 1:N
    asc = (θ̂[1], θ̂[2], θ̂[3], 0.0)
    V = [asc[j] + θ̂[4]*gcost[i,j] + θ̂[5]*wait[i,j] for j in 1:J]
    Phat[i,:] = choice_probabilities(V)
end

car = modeidx["car"];
elasticity_car = (1/N) * sum(θ̂[4]*gcost[i,car]*(1 - Phat[i,car]) for i in 1:N)

println("own-cost elasticity of car = ", round(elasticity_car, digits = 4));

own-cost elasticity of car = -1.0817


## What Did We Estimate?
The cost and terminal-time coefficients are negative: higher generalized cost and longer waits lower a mode's utility and its choice probability. The alternative-specific constants rank the modes' intrinsic appeal relative to `car` after controlling for cost and time. The own-cost elasticity is negative and, in absolute value, larger than one, so mode choice is elastic in cost — a one-percent cost increase lowers the choice probability by more than one percent on average.

> __Key Takeaways.__
>
> * **A logit is a softmax of utilities** — the choice probabilities are $P_{ij}=\exp(V_{ij})/\sum_k\exp(V_{ik})$, and the coefficients come from maximizing the log-likelihood $\sum_i \ln P_{i,c_i}$.
> * **Coefficient signs are interpretable** — negative cost and time coefficients mean cheaper, faster modes are chosen more often.
> * **Elasticities translate coefficients into behavior** — the own-cost elasticity converts $\beta_{\text{gcost}}$ into a percentage response of the choice probability.
___

### Additional Resources
* McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. In P. Zarembka (Ed.), _Frontiers in Econometrics_ (pp. 105–142). Academic Press.
* Greene, W. H. (2018). _Econometric Analysis_ (8th ed.). Pearson. (Source of the `TravelMode` dataset.)
* Train, K. (2009). _Discrete Choice Methods with Simulation_ (2nd ed.). Cambridge University Press.

___